# NDI Task 4: full run and analysis (Colab, checkpointed sessions)

**Task 3 review and Task 4 execution memo, Section 4, steps 7-8.** Start only after ma'am's OK on the pilot. In each Colab session: **Runtime > Run all** and keep the tab open. The script runs C1 (3 levels), C0, C2 (3 levels), then the extension C1b4 if triggered; one file per replicate on Drive, so a disconnect loses at most one replicate per worker. When every cell is complete, F-2 re-runs two replicates (determinism) and writes the analysis, Figure 5, the hand-in check and the deviation note to `results_task4/analysis/`.

- **F-0** Drive, repository at tag `v1.4-task4` (refuses to run untagged); **F-1** full run (`task4_run.py run`); **F-2** progress / determinism / analysis (`task4_run.py verify`, `merge`).
- Another machine can run a different shard at the same time (see F-1); copy its `cells/` files into `results_task4/cells/` on Drive before F-2.

In [ ]:
# F-0 Paths, Drive, and the repository at tag v1.4-task4 (the full run refuses to run untagged).
# Run this notebook in each Colab session (Runtime -> Run all) until F-2 says the run is complete. Keep the tab open.
# G-0 logic: asks once for the read-only token if the private clone fails (hidden prompt, never stored).
import os, sys, json, subprocess, getpass
from google.colab import drive
drive.mount("/content/drive")
PIPELINE_MODULE = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
RESULTS_T4_DIR = "/content/drive/MyDrive/ndi_project/results_task4"      # replicate files persist here
WORKERS = os.cpu_count() or 1           # Colab free: 2
SESSION_HOURS = 11.0                    # no new replicate starts unless it can finish within this budget
REPO_URL = "https://github.com/hpatel09112004/NDI_Depression_Risk_Index.git"
REPO_DIR = "/content/NDI_Depression_Risk_Index"
RUN_TAG = "v1.4-task4"
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

def _git_run(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True, env=_env)
    msg = (r.stderr or "")[-600:]
    return r.returncode, (msg.replace(secret, "***") if secret else msg)

def _token_url():
    tok = getpass.getpass("Private repository: paste the read-only GitHub token (hidden, not saved): ").strip()
    return REPO_URL.replace("https://", f"https://hpatel09112004:{tok}@"), tok

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _rc, _msg = _git_run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])
    if _rc != 0:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "clone", "--quiet", _url, REPO_DIR], secret=_tok)
        assert _rc == 0, "git clone failed (token or URL?):\n" + _msg
        _git_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
else:
    _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet"])
    _have = _git_run(["git", "-C", REPO_DIR, "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0
    if _rc != 0 and not _have:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet", _url], secret=_tok)
        assert _rc == 0, "git fetch failed (token or URL?):\n" + _msg
assert _git_run(["git", "-C", REPO_DIR, "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0, \
    f"tag {RUN_TAG} not found: run NDI_task4_v1_4.ipynb (checks, calibration, tag, pilot) first"
_rc, _msg = _git_run(["git", "-C", REPO_DIR, "checkout", "--quiet", "--force", RUN_TAG])
assert _rc == 0, f"could not check out {RUN_TAG}:\n" + _msg
os.chdir(REPO_DIR)
print("HEAD:", subprocess.run(["git", "log", "-1", "--format=%H %cI %s"], capture_output=True, text=True).stdout.strip())
_pilot = os.path.join(RESULTS_T4_DIR, "pilot_15_replicates.csv")
assert os.path.isfile(_pilot), "the pilot is missing: run NDI_task4_v1_4.ipynb to the end first"
print(f"Pilot present ({_pilot}). Start the full run only after ma'am's OK on the pilot.")
print(f"Workers: {WORKERS}; session budget: {SESSION_HOURS} h; results -> {RESULTS_T4_DIR}")

def run_script(*args):
    cmd = [sys.executable, "-u", os.path.join(REPO_DIR, "task4_run.py"), *args, "--repo", REPO_DIR,
           "--ndi", PIPELINE_MODULE, "--out", RESULTS_T4_DIR]
    print(" ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

In [ ]:
# F-1 Full run (memo, Section 4, step 7): C1 (3 levels), then C0, then C2 (3 levels), then the extension C1b4 if it is
# triggered. One file per replicate (results_task4/cells/<cell>/rep_XXX.json); files that exist are skipped, so after a
# disconnect just Run all again. Another machine can run a different shard at the same time, e.g. on a laptop:
#   python task4_run.py run --ndi ndi_pipeline.py --out results_task4 --cells C2b1,C2b2,C2b3 --workers 8
# (inside the repository checked out at v1.4-task4), then copy its cells/ files into results_task4/cells/ on Drive.
_rc = run_script("run", "--cells", "all", "--reps", "all", "--workers", str(WORKERS), "--hours", str(SESSION_HOURS))
assert _rc == 0, "the run stopped with an error (see above) -- STOP and send the output"

In [ ]:
# F-2 Progress, then (when every cell is complete) the determinism re-run and the analysis (memo, Section 4, step 8):
# per cell power with Wilson 95% CI at P < .05 and P < .0125, mean delta rho +/- Monte Carlo SE, arm means, oracle rho;
# C0 false-positive rate; MDE (probit on cell mean delta rho, C0 + C1 levels, 80% crossing, bootstrap CI over replicates);
# Figure 5; determinism (pilot vs full run, and two replicates re-run in this session); hand-in check; deviation note.
# Outputs: results_task4/analysis/.
_done = run_script("merge")
assert _done == 0, "merge stopped with an error (see above)"
if os.path.isfile(os.path.join(RESULTS_T4_DIR, "analysis", "task4_per_cell.csv")):
    if not os.path.isdir(os.path.join(RESULTS_T4_DIR, "determinism")):
        assert run_script("verify", "--workers", str(min(WORKERS, 2))) == 0, "verify stopped with an error"
        assert run_script("merge") == 0, "merge stopped with an error"
    from IPython.display import Image, display
    display(Image(os.path.join(RESULTS_T4_DIR, "analysis", "figure5_power_curves.png")))
    print(open(os.path.join(RESULTS_T4_DIR, "analysis", "handin_check_task4.txt")).read())
else:
    print("Not complete: Run all again in the next session (completed replicates are kept on Drive).")